# Gate 7: Llama-3.1-8B v2 (train, dev rewrites, GGUF Q6_K)

Rules: `docs/ROADMAP.md`, gate 7. Steps, each saved to Google Drive (`MyDrive/kvkk-rewriter/`) as soon as it exists:
1. QLoRA on `data/pairs_v2.jsonl` with gate 4b's configuration; an adapter after every epoch, the one with the lowest
   validation loss is used (`adapters/qlora_v2_llama3.1-8b`).
2. v2 merged into the bf16 base rewrites the 50 dev questions with the gate 4a decoding.
3. v1 (the gate 4b adapter) rewrites the 20 test_v3 questions, for the final comparison (not scored until the end).
4. v2 merged -> GGUF bf16 -> Q6_K with llama.cpp (`gguf/qlora_v2_llama3.1-8b-Q6_K.gguf`, about 6.6 GB).

If the runtime disconnects, run all again: finished steps are skipped.

Before running:
1. Runtime -> Change runtime type -> **L4 GPU**.
2. Secrets (key icon) -> `HF_TOKEN` (Hugging Face read token with Llama-3.1 access), notebook access on.
3. Runtime -> Run all, and allow Google Drive access when asked (the same account as gate 4b). Keep the tab open.
4. At the end a zip with the result files downloads (adapters and the GGUF stay on Drive). Add the row printed by
   the last cell to `results/colab_usage.csv`, then disconnect and delete the runtime.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert "L4" in gpu, f"expected an L4 runtime (docs/ROADMAP.md, gate 7), got {gpu!r}"

In [ ]:
import os
if not os.path.exists("/content/kvkk-rewriter"):
    subprocess.run(["git", "clone", "-q", "--recurse-submodules",
                    "https://github.com/EmirhanHasirci11/kvkk-rewriter.git", "/content/kvkk-rewriter"], check=True)
%cd /content/kvkk-rewriter
!git pull -q && git log -1 --format='%H %s'
!pip install -q "transformers>=4.56" accelerate peft bitsandbytes
# Colab's preinstalled torchao is older than peft accepts, and loading the adapter then fails; nothing here uses it.
!pip uninstall -q -y torchao

In [ ]:
import json, shutil, sys, time
from pathlib import Path

import torch, transformers, peft, bitsandbytes
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/kvkk-rewriter")
for d in ("results", "adapters", "gguf"):
    (DRIVE / d).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, "scripts")
import qlora, student

NAME = "llama3.1-8b"
HF_ID = student.MODELS[NAME]
PAIRS = Path("data/pairs_v2.jsonl")
ADAPTER_V1 = DRIVE / f"adapters/qlora_v1_{NAME}"
ADAPTER_V2 = DRIVE / f"adapters/qlora_v2_{NAME}"
MERGED = Path(f"/content/merged_qlora_v2_{NAME}")
GGUF = DRIVE / f"gguf/qlora_v2_{NAME}-Q6_K.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
read = lambda p: [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]
dev = read("external/kvkk-rag/eval/golden.jsonl")
test_v3 = read("eval/test_v3.jsonl")
assert len(dev) == 50 and len(test_v3) == 20
warmup = qlora.load_pairs("train", PAIRS)[0]["question"]
LOG = DRIVE / "results/colab_qlora_v2.log"
versions = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
            "bitsandbytes": bitsandbytes.__version__}

# the v1 adapter must be the gate 4b one
on_drive = json.loads((ADAPTER_V1 / "train_record.json").read_text(encoding="utf-8"))
committed = json.loads(Path(f"results/train_record_qlora_v1_{NAME}.json").read_text(encoding="utf-8"))
assert on_drive == committed, "v1 adapter on Drive does not match results/train_record_qlora_v1_llama3.1-8b.json"


def log(msg):
    line = f"{time.strftime('%H:%M:%S')} {msg}"
    print(line)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")


def rewrite_all(model, tok, stop, questions, out, adapter):
    torch.cuda.reset_peak_memory_stats()
    student.rewrite(model, tok, stop, warmup)
    rows = []
    for i, q in enumerate(questions, 1):
        r = student.rewrite(model, tok, stop, q["question"])
        rows.append({"id": q["id"], "question": q["question"], **r, "model": NAME, "hf_id": HF_ID, "adapter": adapter})
        print(f"[{i:2}/{len(questions)}] {r['latency_s']:.2f}s {r['new_tokens']:3} tok{' TRUNC' if r['truncated'] else ''}  {r['rewrite'][:110]}")
    out.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
    lat = sorted(r["latency_s"] for r in rows)
    log(f"{out.name} written; latency p50 {lat[len(lat) // 2]} s, truncated {sum(r['truncated'] for r in rows)}, "
        f"peak {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")


log(f"start: {gpu}, commit {COMMIT[:7]}, {versions}, config {qlora.CONFIG}, pairs {PAIRS}")
t_start = time.time()

In [ ]:
# 1. train v2
if (ADAPTER_V2 / "train_record.json").exists() and "best_epoch" in json.loads((ADAPTER_V2 / "train_record.json").read_text()):
    log("=== v2 adapter already on Drive, not retrained")
else:
    log(f"=== training v2 ({HF_ID}, {PAIRS})")
    record = qlora.train(HF_ID, ADAPTER_V2, token=HF_TOKEN, log=log, pairs=PAIRS, best_val=True)
    record.update({"gpu": gpu, "commit": COMMIT, **versions})
    (ADAPTER_V2 / "train_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    log(f"trained in {record['train_min']} min, val loss {record['val_loss']}, best epoch {record['best_epoch']}, "
        f"peak {record['peak_gpu_gb']} GB")
torch.cuda.empty_cache()

In [ ]:
# 2. v2 dev rewrites, and the merged model kept on local disk for the conversion
out = DRIVE / f"results/rewrites_qlora_v2_{NAME}_dev_v1.jsonl"
if out.exists() and (GGUF.exists() or MERGED.exists()):
    log(f"=== {out.name} already on Drive, skipped")
else:
    model, tok, stop = qlora.load_merged(HF_ID, ADAPTER_V2, token=HF_TOKEN)
    if not out.exists():
        log("=== v2 dev rewrites")
        rewrite_all(model, tok, stop, dev, out, ADAPTER_V2.name)
    if not GGUF.exists() and not MERGED.exists():
        model.save_pretrained(MERGED, max_shard_size="5GB")
        tok.save_pretrained(MERGED)
        log(f"merged v2 saved to {MERGED}")
    del model
    torch.cuda.empty_cache()

In [ ]:
# 3. v1 test_v3 rewrites (scored only at the end, docs/ROADMAP.md gate 7)
out = DRIVE / f"results/rewrites_qlora_v1_{NAME}_test_v3_v1.jsonl"
if out.exists():
    log(f"=== {out.name} already on Drive, skipped")
else:
    log("=== v1 test_v3 rewrites")
    model, tok, stop = qlora.load_merged(HF_ID, ADAPTER_V1, token=HF_TOKEN)
    rewrite_all(model, tok, stop, test_v3, out, ADAPTER_V1.name)
    del model
    torch.cuda.empty_cache()

In [ ]:
# 4. GGUF: merged bf16 -> GGUF bf16 -> Q6_K
def sh(cmd):
    print(">", cmd)
    subprocess.run(cmd, shell=True, check=True)

if GGUF.exists():
    log(f"=== {GGUF.name} already on Drive, skipped")
else:
    if not Path("/content/llama.cpp").exists():
        sh("git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp")
    lc_commit = subprocess.run(["git", "-C", "/content/llama.cpp", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    sh("pip install -q /content/llama.cpp/gguf-py")
    sh("cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null")
    sh("cmake --build /content/llama.cpp/build --target llama-quantize -j 8 > /dev/null")
    assert MERGED.exists(), "run step 2 again: the merged model is missing"
    bf16 = Path(f"/content/qlora_v2_{NAME}-bf16.gguf")
    local_q6 = Path("/content") / GGUF.name
    sh(f"python /content/llama.cpp/convert_hf_to_gguf.py {MERGED} --outtype bf16 --outfile {bf16}")
    sh(f"/content/llama.cpp/build/bin/llama-quantize {bf16} {local_q6} Q6_K")
    shutil.copy(local_q6, GGUF)
    log(f"{GGUF.name} on Drive: {GGUF.stat().st_size / 1e9:.2f} GB, llama.cpp {lc_commit[:10]}")
log(f"done in {(time.time() - t_start) / 60:.1f} min this session")

In [ ]:
import zipfile
from google.colab import files
paths = sorted((DRIVE / "results").glob("rewrites_qlora_v2_*_dev_v1.jsonl")) + \
        sorted((DRIVE / "results").glob("rewrites_qlora_v1_*_test_v3_v1.jsonl")) + [LOG]
with zipfile.ZipFile("qlora_v2.zip", "w") as z:
    for p in paths:
        z.write(p, f"results/{p.name}")
    z.write(ADAPTER_V2 / "train_record.json", f"results/train_record_qlora_v2_{NAME}.json")
files.download("qlora_v2.zip")
print("row for results/colab_usage.csv (fill in runtime_min and compute_units from Colab):")
print(f"{time.strftime('%Y-%m-%d')},notebooks/04_qlora_v2.ipynb,{gpu},<runtime_min>,<compute_units>,v2 training + dev rewrites + v1 test_v3 rewrites + GGUF Q6_K,commit {COMMIT[:7]}")